In [1]:
from pyfaidx import Fasta

In [2]:
genome = Fasta(
    "../data/reference/hg38.fa"
)

In [3]:
genome

Fasta("../data/reference/hg38.fa")

In [4]:
sequence = genome["chr1"][1000:1100].seq

print(sequence)

NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN


In [6]:
from pybigtools import open

bb = open(
    "../data/raw/ENCFF106WBX.bigBed"
)

In [7]:
print(bb.chroms())

{'chr1': 248956422, 'chr10': 133797422, 'chr11': 135086622, 'chr12': 133275309, 'chr13': 114364328, 'chr14': 107043718, 'chr15': 101991189, 'chr16': 90338345, 'chr17': 83257441, 'chr18': 80373285, 'chr19': 58617616, 'chr2': 242193529, 'chr20': 64444167, 'chr21': 46709983, 'chr22': 50818468, 'chr3': 198295559, 'chr4': 190214555, 'chr5': 181538259, 'chr6': 170805979, 'chr7': 159345973, 'chr8': 145138636, 'chr9': 138394717, 'chrX': 156040895, 'chrY': 57227415}


In [11]:
records = list(
    bb.records("chr1", 0, 10_000_000)
)

print(len(records))
print(records[:5])

663
[(778479, 779226, '.', '1000', '.', '26.93655', '713.72321', '709.58307', '299'), (778479, 779226, '.', '1000', '.', '3.62514', '25.43981', '23.19857', '647'), (817132, 817914, '.', '1000', '.', '6.20372', '70.13375', '67.57131', '304'), (826649, 828088, '.', '1000', '.', '11.76791', '588.28070', '584.43811', '940'), (826649, 828088, '.', '1000', '.', '2.16284', '17.48284', '15.34591', '400')]


In [12]:
first_record = records[0]

print(first_record)
"""structure is: (
  start,
  end,
  name,
  score,
  strand,
  signalValue,
  pValue,
  qValue,
  peak
)"""
print(type(first_record))

(778479, 779226, '.', '1000', '.', '26.93655', '713.72321', '709.58307', '299')
<class 'tuple'>


In [13]:
import pandas as pd

rows = []

for record in records:

    start, end, name, score, strand, signalValue, pValue, qValue, peak = record

    rows.append({
        "chrom": "chr1",
        "start": start,
        "end": end,
        "signalValue": float(signalValue),
        "pValue": float(pValue),
        "qValue": float(qValue)
    })

df = pd.DataFrame(rows)

print(df.head())

  chrom   start     end  signalValue     pValue     qValue
0  chr1  778479  779226     26.93655  713.72321  709.58307
1  chr1  778479  779226      3.62514   25.43981   23.19857
2  chr1  817132  817914      6.20372   70.13375   67.57131
3  chr1  826649  828088     11.76791  588.28070  584.43811
4  chr1  826649  828088      2.16284   17.48284   15.34591


In [14]:
row = df.iloc[0]

sequence = genome[
    row["chrom"]
][
    row["start"]:row["end"]
].seq

print(sequence[:100])
print(len(sequence))

CACTGTGCTGGGTCTGAATTTTTCAGCTACCCTTCTTCAGCCGGCAACACACAGAACCTGGCGGGGAGGTCACTCTTACCAGTCCCCACTCTGATGAGAA
747


In [15]:
WINDOW_SIZE = 200


def extract_fixed_window(
    genome,
    chrom,
    start,
    peak_offset,
    window_size=WINDOW_SIZE
):

    summit = start + peak_offset

    half_window = window_size // 2

    window_start = summit - half_window
    window_end = summit + half_window

    sequence = genome[
        chrom
    ][
        window_start:window_end
    ].seq.upper()

    return sequence

In [16]:
positive_sequences = []

for record in records[:5000]:

    (
        start,
        end,
        name,
        score,
        strand,
        signalValue,
        pValue,
        qValue,
        peak
    ) = record

    sequence = extract_fixed_window(
        genome=genome,
        chrom="chr1",
        start=start,
        peak_offset=int(peak)
    )

    if len(sequence) == 200:

        positive_sequences.append({
            "sequence": sequence,
            "label": 1
        })

In [17]:
positive_df = pd.DataFrame(
    positive_sequences
)

print(positive_df.head())
print(len(positive_df))

                                            sequence  label
0  CCGCCCTTGTGACGTCACGGAAGGCGCACCCTTGTGACGTCACAGG...      1
1  CAGAAGCTCCTCAATGGCCAGCGCCAGCTGCAGCCCCGGCCGCCCA...      1
2  GAAAAGGGAAGAATGCAAAAGTCAAAGACACGTCACCCTCCTTGAG...      1
3  GGACCTTGGCTCCGGATAATCCGTTTCCGGGTCAACAAAAAACGTC...      1
4  GGGAGGCGATTGTGCAGAAGCACGAGGGTTGTTACAGGATCGGGCA...      1
663


In [18]:
print(positive_df.iloc[0]["sequence"])
print(len(positive_df.iloc[0]["sequence"]))

CCGCCCTTGTGACGTCACGGAAGGCGCACCCTTGTGACGTCACAGGGGACTACCACTCACGCAGAGCCAATCAGAACTCGCGGTGGGGGCTGCTGGTTCTTCCAGGAGCGCGCATGAGCGGACGCTGCCTACTGGTGGCCGGGCGGGATGTAACCGGCTGCTGAGCTGGCAGTTCTGTGTCGCTAGGCTTCTGCCCGGCC
200


In [19]:
from collections import Counter

sequence = positive_df.iloc[0]["sequence"]

print(Counter(sequence))

Counter({'G': 69, 'C': 62, 'T': 38, 'A': 31})


In [20]:
peak_intervals = []

for record in records:

    (
        start,
        end,
        name,
        score,
        strand,
        signalValue,
        pValue,
        qValue,
        peak
    ) = record

    peak_intervals.append(
        (start, end)
    )

In [21]:
def overlaps_peak(
    query_start,
    query_end,
    peak_intervals
):

    for peak_start, peak_end in peak_intervals:

        if (
            query_start < peak_end
            and
            query_end > peak_start
        ):
            return True

    return False

In [22]:
import random

negative_sequences = []

CHROM_SIZE = bb.chroms()["chr1"]

while len(negative_sequences) < 5000:

    random_start = random.randint(
        0,
        CHROM_SIZE - 200
    )

    random_end = random_start + 200

    if overlaps_peak(
        random_start,
        random_end,
        peak_intervals
    ):
        continue

    sequence = genome[
        "chr1"
    ][
        random_start:random_end
    ].seq.upper()

    if len(sequence) != 200:
        continue

    negative_sequences.append({
        "sequence": sequence,
        "label": 0
    })

In [23]:
negative_df = pd.DataFrame(
    negative_sequences
)

print(negative_df.head())
print(len(negative_df))

                                            sequence  label
0  NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...      0
1  CTAAGTGCTTCAGGACATAATGATTCAGAGAAGCTAGGATGCTTGA...      0
2  CCCAGTGTTCCCCTGGGGGCAAAATCACCCCTGGTTGAGCATCATT...      0
3  TGAGGTGGAAGGATCACCTGAGCCCAGGAGGTCGAGGCTGTAGTGA...      0
4  ATCCTTTATTTCCTTCTCCTGCCTGATTACCCTGGCCAGAACTTTC...      0
5000


In [24]:
dataset_df = pd.concat(
    [positive_df, negative_df],
    ignore_index=True
)

dataset_df = dataset_df.sample(
    frac=1
).reset_index(drop=True)

print(dataset_df.head())

                                            sequence  label
0  CTGAAGTCCAAGCTTCTTAAACACCAAGACAAAGAGGGACATAATC...      0
1  AGATGCTCTTTGCCAGGCCCTGACTCTGCGAGCTCATTCTTCCATC...      0
2  AAAATATTATACTTTAGGCTGTTTCTGCATCATGCAAAATATTCCT...      0
3  AGGGATCAGAGGTGACAGAATCGTGTATTTCTACATTTATCTGCGG...      0
4  TCAAGGTAAGATTTGGGCAGGGACACAGAGACAAACCATATCATTT...      0


In [25]:
dataset_df.to_csv(
    "../data/processed/liver_accessibility.csv",
    index=False
)